# Assessment corpus italia-corpus — fork Lab

**Repo**: `dataciviclab/italia-corpus`  
**Upstream**: `ahmeabd/italia-corpus` (288.287 file, 23 collezioni)  
**Fork Lab**: 20 collezioni selezionate, sync selettivo  

Obiettivo: valutare se le 20 collezioni del fork coprono le esigenze delle analisi "legge dice, dati mostrano", e dove ci sono buchi.

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter

REPO = Path.cwd()
if REPO.name == 'notebooks':
    REPO = REPO.parent

df = pd.read_parquet(REPO / 'data' / 'derived' / 'normativa.parquet')
print(f'Atti unici nel fork: {len(df):,}')
print(f'Colonne: {list(df.columns)}')

Atti unici nel fork: 20,711
Colonne: ['collezione', 'filename', 'tipo', 'data', 'numero', 'oggetto', 'entrata_vigore', 'celex', 'anno_atto', 'anno_dir', 'ritardo']


In [2]:
# Separa le collezioni multiple (un atto può stare in più collezioni)
df['collezioni_list'] = df['collezione'].str.split(';')

# Esplodi: una riga per (atto, collezione)
df_exploded = df.explode('collezioni_list')
print(f'Righe dopo explode (atto×collezione): {len(df_exploded):,}')
print(f'Collezioni distinte: {df_exploded["collezioni_list"].nunique()}')

Righe dopo explode (atto×collezione): 25,628
Collezioni distinte: 20


## 2. Copertura per collezione

In [3]:
# Atti unici per collezione
col_counts = df_exploded.groupby('collezioni_list').size().sort_values(ascending=False)
print(f"{'Collezione':55s} | {'Atti':>6s} | {'%':>5s}")
print('-' * 70)
total = len(df)
for col, count in col_counts.items():
    print(f'{col:55s} | {count:>6,} | {count/total*100:>4.1f}%')
print(f'\nTotale atti unici: {total:,}')

Collezione                                              |   Atti |     %
----------------------------------------------------------------------
DL e leggi di conversione                               |  9,740 | 47.0%
Decreti Legislativi                                     |  2,898 | 14.0%
Leggi di ratifica                                       |  2,328 | 11.2%
Regolamenti ministeriali                                |  2,077 | 10.0%
Regolamenti governativi                                 |  1,940 |  9.4%
DL decaduti                                             |  1,735 |  8.4%
Decreti legislativi luogotenenziali                     |  1,216 |  5.9%
Leggi delega e relativi provvedimenti delegati          |  1,135 |  5.5%
Atti di recepimento direttive UE                        |  1,085 |  5.2%
Regolamenti di delegificazione                          |    375 |  1.8%
DPCM                                                    |    359 |  1.7%
Testi Unici                                          

## 3. Copertura temporale

In [4]:
# Converte anno_atto in int, scarta null
df_year = df.dropna(subset=['anno_atto']).copy()
df_year['anno'] = df_year['anno_atto'].astype(int)
df_year = df_year[(df_year['anno'] >= 1860) & (df_year['anno'] <= 2026)]

print('Atti per decennio:')
df_year['decennio'] = (df_year['anno'] // 10) * 10
dec = df_year.groupby('decennio').size()
for d, c in dec.items():
    print(f'  {d}-{d+9}: {c:>6,} atti')
print(f'\nPeriodo coperto: {df_year["anno"].min()} - {df_year["anno"].max()}')

Atti per decennio:
  1870-1879:     10 atti
  1880-1889:     19 atti
  1890-1899:     20 atti
  1900-1909:     40 atti
  1910-1919:     27 atti
  1920-1929:  1,571 atti
  1930-1939:  3,549 atti
  1940-1949:  2,555 atti
  1950-1959:    654 atti
  1960-1969:    536 atti
  1970-1979:    822 atti
  1980-1989:  1,516 atti
  1990-1999:  4,195 atti
  2000-2009:  2,454 atti
  2010-2019:  1,731 atti
  2020-2029:  1,012 atti

Periodo coperto: 1874 - 2026


In [5]:
# Heatmap: collezione × decennio
# Usa groupby invece di crosstab per evitare duplicati
df_heat = df_exploded.dropna(subset=['anno_atto']).copy()
df_heat['decennio'] = (df_heat['anno_atto'].astype(float) // 10 * 10).astype(int)
ct = df_heat.groupby(['collezioni_list', 'decennio']).size().unstack(fill_value=0)
display(ct)
# Salva per export
ct.to_csv(REPO / 'data' / 'derived' / 'assessment_copertura.csv')
print(f'Heatmap: {ct.shape[0]} collezioni × {ct.shape[1]} decenni')

decennio,1870,1880,1890,1900,1910,1920,1930,1940,1950,1960,1970,1980,1990,2000,2010,2020
collezioni_list,,,,,,,,,,,,,,,,
Atti di attuazione Regolamenti UE,0,0,0,0,0,0,0,0,0,0,1,16,19,1,1,1
Atti di recepimento direttive UE,0,0,0,0,0,0,0,0,0,2,12,125,314,290,237,105
Codici,0,0,0,0,0,0,1,5,1,0,1,3,3,11,11,2
DL decaduti,0,0,0,0,0,0,0,0,3,10,68,388,1185,51,30,0
DL e leggi di conversione,0,0,1,10,8,1521,3500,354,434,293,541,724,911,684,447,312
DL proroghe,0,0,0,0,0,0,0,0,0,0,1,18,15,22,11,8
DPCM,0,0,0,0,0,0,0,0,0,0,0,22,91,64,107,75
Decreti Legislativi,0,0,0,0,0,0,0,807,0,0,0,26,596,619,567,283
Decreti legislativi luogotenenziali,0,0,0,0,0,0,0,1216,0,0,0,0,0,0,0,0


Heatmap: 20 collezioni × 16 decenni


## 4. Atti recenti (2020-2026)

In [6]:
recenti = df_year[df_year['anno'] >= 2020]

print(f'Atti dal 2020: {len(recenti):,}')
print(f'Copertura {recenti["anno"].min()} - {recenti["anno"].max()}')
print()

# Per anno
for anno in range(2020, 2027):
    count = len(recenti[recenti['anno'] == anno])
    print(f'  {anno}: {count} atti')

Atti dal 2020: 1,012
Copertura 2020 - 2026

  2020: 151 atti
  2021: 183 atti
  2022: 155 atti
  2023: 170 atti
  2024: 156 atti
  2025: 141 atti
  2026: 56 atti


In [7]:
# Atti recenti per collezione principale
recent_exploded = df_exploded[
    df_exploded['anno_atto'].notna() &
    (df_exploded['anno_atto'].astype(int) >= 2020)
]
print('Atti 2020-2026 per collezione:')
for col, count in recent_exploded.groupby('collezioni_list').size().sort_values(ascending=False).items():
    print(f'  {col:50s} {count}')

Atti 2020-2026 per collezione:
  DL e leggi di conversione                          312
  Regolamenti ministeriali                           283
  Decreti Legislativi                                283
  Leggi delega e relativi provvedimenti delegati     260
  Leggi di ratifica                                  145
  Atti di recepimento direttive UE                   105
  DPCM                                               75
  Regolamenti governativi                            54
  Leggi contenenti deleghe                           25
  Regolamenti di delegificazione                     13
  DL proroghe                                        8
  Testi Unici                                        6
  Leggi costituzionali                               6
  Leggi finanziarie e di bilancio                    6
  Leggi di delegazione europea                       5
  Codici                                             2
  Atti di attuazione Regolamenti UE                  1


## 5. Dedup: atti in multiple collezioni

In [8]:
df['num_collezioni'] = df['collezioni_list'].str.len()

print('Distribuzione atti per numero di collezioni:')
for n in sorted(df['num_collezioni'].unique()):
    count = (df['num_collezioni'] == n).sum()
    print(f'  {n} collezione/i: {count:>6,} atti ({count/len(df)*100:.1f}%)')

print(f'\nAtti in 3+ collezioni ("omnibus"): {(df["num_collezioni"] >= 3).sum():,}')
print(f'Atti in 5+ collezioni: {(df["num_collezioni"] >= 5).sum():,}')

Distribuzione atti per numero di collezioni:
  1 collezione/i: 16,111 atti (77.8%)
  2 collezione/i:  4,286 atti (20.7%)
  3 collezione/i:    311 atti (1.5%)
  4 collezione/i:      3 atti (0.0%)

Atti in 3+ collezioni ("omnibus"): 314
Atti in 5+ collezioni: 0


## 6. Qualità metadati

In [9]:
print(f"{'Campo':25s} | {'Compilati':>10s} | {'%':>5s} | {'Vuoti':>8s}")
print('-' * 54)
for col in df.columns:
    if col in ('collezioni_list', 'num_collezioni'):
        continue
    total = len(df)
    if col == 'ritardo':
        nulls = df[col].isna().sum()
    else:
        nulls = (df[col].isna() | (df[col].astype(str).str.strip() == '')).sum()
    filled = total - nulls
    print(f'{col:25s} | {filled:>10,} | {filled/total*100:>4.1f}% | {nulls:>8,}')

print(f'\nTotale atti: {len(df):,}')

Campo                     |  Compilati |     % |    Vuoti
------------------------------------------------------


collezione                |     20,711 | 100.0% |        0
filename                  |     20,711 | 100.0% |        0
tipo                      |     20,711 | 100.0% |        0
data                      |     20,711 | 100.0% |        0
numero                    |     20,711 | 100.0% |        0
oggetto                   |     20,711 | 100.0% |        0
entrata_vigore            |     11,543 | 55.7% |    9,168
celex                     |      2,867 | 13.8% |   17,844
anno_atto                 |     20,711 | 100.0% |        0
anno_dir                  |     20,711 | 100.0% |        0
ritardo                   |      2,631 | 12.7% |   18,080

Totale atti: 20,711


In [10]:
# Qualità per collezione: % oggetto compilato
qual = df_exploded.groupby('collezioni_list').agg(
    totale=('filename', 'count'),
    con_oggetto=('oggetto', lambda x: x.notna().sum()),
    con_celex=('celex', lambda x: x.notna().sum()),
    con_vigore=('entrata_vigore', lambda x: x.notna().sum()),
).reset_index()
qual['%_oggetto'] = (qual['con_oggetto'] / qual['totale'] * 100).round(1)
qual['%_celex'] = (qual['con_celex'] / qual['totale'] * 100).round(1)
print(qual.sort_values('%_oggetto').to_string(index=False))

                               collezioni_list  totale  con_oggetto  con_celex  con_vigore  %_oggetto  %_celex
             Atti di attuazione Regolamenti UE      39           39         39          39      100.0    100.0
              Atti di recepimento direttive UE    1085         1085       1085        1085      100.0    100.0
                                        Codici      38           38         38          38      100.0    100.0
                                   DL decaduti    1735         1735       1735        1735      100.0    100.0
                     DL e leggi di conversione    9740         9740       9740        9740      100.0    100.0
                                   DL proroghe      75           75         75          75      100.0    100.0
                                          DPCM     359          359        359         359      100.0    100.0
                           Decreti Legislativi    2898         2898       2898        2898      100.0    100.0
 

## 7. Recepimento direttive UE

In [11]:
rece = df_exploded[df_exploded['collezioni_list'] == 'Atti di recepimento direttive UE'].copy()
rece = rece[rece['anno_atto'].notna() & (rece['anno_atto'].astype(int) >= 1990)]
rece['anno'] = rece['anno_atto'].astype(int)

print('Recepimenti UE per anno:')
for anno in range(1990, 2027):
    count = (rece['anno'] == anno).sum()
    if count > 0:
        print(f'  {anno}: {count}')

Recepimenti UE per anno:
  1990: 3
  1991: 24
  1992: 97
  1993: 22
  1994: 19
  1995: 13
  1996: 32
  1997: 32
  1998: 33
  1999: 39
  2000: 37
  2001: 23
  2002: 14
  2003: 46
  2004: 26
  2005: 40
  2006: 25
  2007: 50
  2008: 17
  2009: 12
  2010: 44
  2011: 33
  2012: 20
  2013: 5
  2014: 28
  2015: 18
  2016: 41
  2017: 18
  2018: 22
  2019: 8
  2020: 21
  2021: 30
  2022: 6
  2023: 12
  2024: 11
  2025: 13
  2026: 12


In [12]:
# Ritardo recepimento (anni tra direttiva e atto nazionale)
con_ritardo = rece[rece['ritardo'].notna()].copy()
con_ritardo['ritardo_num'] = con_ritardo['ritardo'].astype(float)

print(f'Recepimenti con ritardo calcolabile: {len(con_ritardo)}')
print(f'Ritardo medio: {con_ritardo["ritardo_num"].mean():.1f} anni')
print(f'Ritardo mediano: {con_ritardo["ritardo_num"].median():.1f} anni')
print()

# Per decennio
con_ritardo['decennio'] = (con_ritardo['anno'] // 10) * 10
for dec, group in con_ritardo.groupby('decennio'):
    print(f'{dec}-{dec+9}: media {group["ritardo_num"].mean():.1f} anni, mediana {group["ritardo_num"].median():.1f} anni (n={len(group)})')

Recepimenti con ritardo calcolabile: 727
Ritardo medio: 3.5 anni


Ritardo mediano: 3.0 anni

1990-1999: media 3.1 anni, mediana 3.0 anni (n=142)
2000-2009: media 2.5 anni, mediana 2.0 anni (n=247)
2010-2019: media 3.0 anni, mediana 3.0 anni (n=233)
2020-2029: media 7.7 anni, mediana 4.0 anni (n=105)


## 8. Verifica atti mancanti

In [13]:
# Lista di atti "famosi" da verificare
atti_cercati = [
    ('D.Lgs 199/2021', '199'),
    ('D.Lgs 28/2011', '28'),
    ('D.Lgs 387/2003', '387'),
    ('D.Lgs 149/2022', '149'),  # riforma Cartabia
    ('L. 107/2015', '107'),      # Buona Scuola
    ('L. 124/2015', '124'),      # riforma Madia
    ('D.Lgs 152/2006', '152'),   # TUA ambiente
    ('D.Lgs 50/2016', '50'),     # Codice appalti
    ('D.Lgs 36/2023', '36'),     # Nuovo codice appalti
    ('L. 833/1978', '833'),      # Istituzione SSN
]

print(f"{'Atto':25s} | {'Cerca':>10s} | {'Trovato?':>10s} | {'Collezione'}")
print('-' * 80)

for nome, num in atti_cercati:
    # Cerca per numero nell'oggetto o nel filename
    match = df[
        df['oggetto'].str.contains(f'n\.\s*{num}[^\d]|n\.{num}|{num}$', na=False, regex=True) |
        df['filename'].str.contains(f'n\.\s*{num}|\b{num}\b', na=False, regex=True)
    ]
    if len(match) > 0:
        print(f"{nome:25s} | {num:>10s} | {'✅ SÌ':>10s} | {match.iloc[0]['collezione'][:50]}")
    else:
        print(f"{nome:25s} | {num:>10s} | {'❌ NO':>10s} | -")

Atto                      |      Cerca |   Trovato? | Collezione
--------------------------------------------------------------------------------


D.Lgs 199/2021            |        199 |       ✅ SÌ | Atti di recepimento direttive UE;Regolamenti gover
D.Lgs 28/2011             |         28 |       ✅ SÌ | Atti di attuazione Regolamenti UE
D.Lgs 387/2003            |        387 |       ✅ SÌ | DL e leggi di conversione


D.Lgs 149/2022            |        149 |       ✅ SÌ | DL e leggi di conversione


L. 107/2015               |        107 |       ✅ SÌ | DL decaduti


<>:21: SyntaxWarning: invalid escape sequence '\.'
<>:21: SyntaxWarning: invalid escape sequence '\d'
<>:22: SyntaxWarning: invalid escape sequence '\.'
<>:21: SyntaxWarning: invalid escape sequence '\.'
<>:21: SyntaxWarning: invalid escape sequence '\d'
<>:22: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_50831/3001234056.py:21: SyntaxWarning: invalid escape sequence '\.'
  df['oggetto'].str.contains(f'n\.\s*{num}[^\d]|n\.{num}|{num}$', na=False, regex=True) |
/tmp/ipykernel_50831/3001234056.py:21: SyntaxWarning: invalid escape sequence '\d'
  df['oggetto'].str.contains(f'n\.\s*{num}[^\d]|n\.{num}|{num}$', na=False, regex=True) |
/tmp/ipykernel_50831/3001234056.py:22: SyntaxWarning: invalid escape sequence '\.'
  df['filename'].str.contains(f'n\.\s*{num}|\b{num}\b', na=False, regex=True)


L. 124/2015               |        124 |       ✅ SÌ | Codici;Decreti Legislativi


D.Lgs 152/2006            |        152 |       ✅ SÌ | Atti di attuazione Regolamenti UE


D.Lgs 50/2016             |         50 |       ✅ SÌ | DL decaduti
D.Lgs 36/2023             |         36 |       ✅ SÌ | DL e leggi di conversione


L. 833/1978               |        833 |       ✅ SÌ | DL e leggi di conversione


In [14]:
# Buchi recenti: cerchiamo D.Lgs che iniziano con "21G" (2021) o "22G" (2022) nell'upstream
# ma non nel fork (se abbiamo accesso all'upstream)
try:
    import subprocess
    # Lista D.Lgs nel fork per anno
    fork_dlgs = set(df_exploded[
        df_exploded['collezioni_list'] == 'Decreti Legislativi'
    ]['filename'])
    print(f'D.Lgs nel fork: {len(fork_dlgs)}')
    
    # Confronto upstream per i filename
    result = subprocess.run(
        ['git', 'ls-tree', 'upstream/main', '--', 'Decreti Legislativi/'],
        capture_output=True, text=True, cwd=REPO
    )
    upstream_files = set()
    for line in result.stdout.strip().split('\n'):
        if line:
            parts = line.split('\t')
            if len(parts) >= 2:
                upstream_files.add(parts[-1].replace('Decreti Legislativi/', ''))
    
    print(f'D.Lgs nell\'upstream: {len(upstream_files)}')
    
    solo_upstream = upstream_files - fork_dlgs
    solo_fork = fork_dlgs - upstream_files
    print(f'\nSolo upstream (non nel fork): {len(solo_upstream)}')
    if solo_upstream:
        for f in sorted(list(solo_upstream))[:10]:
            print(f'  + {f}')
        if len(solo_upstream) > 10:
            print(f'  ... e altri {len(solo_upstream)-10}')
    print(f'\nSolo fork (non upstream): {len(solo_fork)}')
except Exception as e:
    print(f'Errore nel confronto: {e}')

D.Lgs nel fork: 2898


D.Lgs nell'upstream: 2911

Solo upstream (non nel fork): 33
  + "Norma di attuazione dello statuto speciale per la regione Trentino-Alto Adige-S\303\274dtirol recante modificazioni del decreto del Presidente della Repubblica 1 febbraio 1973 n. 49 Norme di at_20c5a40e0638.md"
  + "Norma di attuazione dello statuto speciale per la regione Trentino-Alto Adige-S\303\274dtirol recante modifiche e integrazioni al decreto del Presidente della Repubblica 26 luglio 1976 n. 752 N_6d92cabb1047.md"
  + "Norme di attuazione dello Statuto speciale della Regione Trentino-Alto Adige-S\303\274dtirol in materia di accademia di belle arti istituti superiori per le industrie artistiche conservatori di_66f09b4bb1ad.md"
  + "Norme di attuazione dello Statuto speciale della Regione Trentino-Alto Adige-S\303\274dtirol in materia di accademia di belle arti istituti superiori per le industrie artistiche conservatori di_73a9cb22ea40.md"
  + "Norme di attuazione dello Statuto speciale per il Trentino-Alto Adige-S

## 9. Verdetto

Basandosi sui dati qui sopra, valutare:

1. **Copertura temporale**: ci sono buchi in periodi chiave?
2. **Atti recenti**: copertura sufficiente per analisi su dati 2000+?
3. **Qualità metadati**: campi utili (oggetto, CELEX, vigore) sono compilati?
4. **Recepimento UE**: quanti recepimenti abbiamo, con che ritardo?
5. **Atti mancanti**: quanti atti famosi non ci sono?
6. **Confronto upstream**: quegli atti mancano anche lì?

Conclusioni: l'80/20 del fork Lab è adeguato per le analisi "legge dice, dati mostrano"?
Se ci sono buchi, sono accettabili o servono interventi?